# Testing the data source

The job ads for this project come from the job search of the Bundesagentur für
Arbeit, the largest database of open positions in Germany.

There is no official public API. However, the community project
[bundesAPI](https://github.com/bundesAPI/jobsuche-api) has documented how the
website and the app of the Bundesagentur retrieve their data. This notebook
tests whether that access works and what the data looks like.

First attempt:

In [ ]:
import requests

url = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/jobs"
headers = {"X-API-Key": "jobboerse-jobsuche"}
params = {"was": "Data Analyst", "size": 25, "page": 1}

response = requests.get(url, headers=headers, params=params)
print(response.status_code)
data = response.json()
data.keys()

**Result: status 403 (forbidden).** The server understood the request but
refused it. This attempt used the older `v4` endpoint and sent only the API key.

Second attempt:

In [ ]:
import requests

url = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v6/jobs"
headers = {
    "X-API-Key": "jobboerse-jobsuche",
    "User-Agent": "Jobsuche/2.9.2 (de.arbeitsagentur.jobboerse; build:1077; iOS 15.1.0) Alamofire/5.4.4",
}
params = {"was": "Data Analyst", "size": 25, "page": 1}

response = requests.get(url, headers=headers, params=params)
print(response.status_code)
print(response.headers.get("Content-Type"))
print(response.text[:500])   # first 500 characters of the answer

**Result: status 200, and the answer is JSON.** The request worked. Two things
were changed compared with the first attempt:

- the newer `v6` endpoint, which the bundesAPI documentation now lists
- a `User-Agent` header, so the request identifies itself like the official app

Both changes were made at once, so it is not clear which one was decisive. For
this project it is enough that the combination works.

### Let's see what a read job ad from the `v6` endpoint contains in detail.

In [ ]:
data = response.json()
print(data.keys())
print(len(data["ergebnisliste"]))
data["ergebnisliste"][0].keys()

The answer has five parts. The ads themselves are in `ergebnisliste`. The other parts describe the search: the total number of results
(`maxErgebnisse`), the current page and page size, and filter options
(`facetten`).

This page contains 25 ads, as requested with `size`. Each ad has 17 fields,
including the title, employer, location, home office option, contract type,
publication date and occupation categories.

### How many ads exist, and what does one look like?

In [ ]:
print(data["maxErgebnisse"])          # how many Data Analyst ads exist in total
data["ergebnisliste"][0]              # one full ad, to see the values

There are **371 ads** for "Data Analyst" alone. Together with related searches
such as "Data Scientist" or "Data Engineer", the dataset should reach one to two
thousand ads, which is a good size for this project.

The example ad shows what the fields contain: a Data Analyst position at
micronex GmbH in Springe (Niedersachsen), full-time, permanent, home office by
arrangement, no salary information, published on 28 September 2026. The location
even comes with coordinates, which would allow a map later.

**What is missing:** the job description itself. The search only returns a
summary of each ad. For extracting skills, the full text is needed.

### Fetching the full text of one ad

In [ ]:
import base64

refnr = data["ergebnisliste"][0]["referenznummer"]
encoded = base64.b64encode(refnr.encode()).decode()

detail_url = f"https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/jobdetails/{encoded}"
detail = requests.get(detail_url, headers=headers)

print(detail.status_code)
print(detail.text[:1000])

**Result: status 200 — the full text is available.** It comes from a second
endpoint, the job details, which is queried with the reference number of each
ad. The documentation requires that number to be Base64-encoded.

The description is stored in `stellenangebotsBeschreibung`. It is formatted text
with clear section headings, such as *Ihre Aufgaben* (the tasks) and *Was wir
von Ihnen erwarten* (the requirements). The second section is where the required
skills are listed — in this example Power BI, SQL, Power Query, DAX and SAP.
That structure will make the skill extraction considerably easier.

## Summary

The data source works:

- The search returns structured information for each ad, and 371 ads for "Data
  Analyst" alone.
- The full description is available through a second request per ad.
- The descriptions are structured, with a separate section for requirements.

**Points to keep in mind for the data collection:**

- Full texts require one request per ad. With hundreds of ads, a short pause
  between requests is needed so as not to overload the server or get blocked.
- The API is unofficial and may change without notice. This is a known risk of
  the data source, and the collection code should handle failed requests.